In [38]:
"""
CASE STUDY 2 — Credit Card Fraud Detection (SIMPLE VERSION, no functions)
=============================================================================

Same pipeline as before (XGBoost + SVM baseline, SMOTE for imbalance),
just written top-to-bottom as plain steps instead of functions.

"""

'\nCASE STUDY 2 — Credit Card Fraud Detection (SIMPLE VERSION, no functions)\n=============================================================================\n\nSame pipeline as before (XGBoost + SVM baseline, SMOTE for imbalance),\njust written top-to-bottom as plain steps instead of functions.\n \n'

In [39]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

In [40]:
# ============================================================================
# STEP 1: DATA LOAD
# ============================================================================
df = pd.read_csv("creditcard.csv")
print("Loaded rows:", len(df), " columns:", df.shape[1])

Loaded rows: 284807  columns: 31


In [41]:
# ============================================================================
# STEP 2: PREPROCESSING -> DATA CLEANING
# ============================================================================
print("\nRows before cleaning:", len(df))
print("Missing values per column (only shown if >0):")
missing = df.isna().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None found.")

n_dupes = df.duplicated().sum()
print("Duplicate rows:", n_dupes)
df = df.drop_duplicates().reset_index(drop=True)
print("Rows after dropping duplicates:", len(df))

fraud_rate = df["Class"].mean()
print("Fraud rate:", round(fraud_rate, 5), " (", df["Class"].sum(), "fraud /", len(df), "total )")

# 'Time' is just seconds-since-start, not very meaningful on its own.
# Convert it into 'Hour' (0-23, hour of day) instead, and drop raw Time.
df["Hour"] = (df["Time"] // 3600) % 24
df = df.drop(columns=["Time"])


Rows before cleaning: 284807
Missing values per column (only shown if >0):
None found.
Duplicate rows: 1081
Rows after dropping duplicates: 283726
Fraud rate: 0.00167  ( 473 fraud / 283726 total )


In [42]:
# ============================================================================
# STEP 3: ML MODEL SETUP (XGBoost main model + SVM baseline)
# ============================================================================
target = "Class"
feature_names = [c for c in df.columns if c != target]
X = df[feature_names]
y = df[target]

# stratify=y keeps the same tiny fraud rate in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print("\nTrain size:", len(X_train), " Test size:", len(X_test))
print("Fraud rate — train:", round(y_train.mean(), 5), " test:", round(y_test.mean(), 5))

# Scale all features (Time/Amount are unscaled, V1-V28 are already PCA-scaled
# but scaling everything together doesn't hurt and keeps things simple)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)   # test set: scaled, never resampled

# SMOTE balances the training classes (only the training set — never test,
# that would leak information and give a falsely optimistic score)
print("\nClass counts before SMOTE:", dict(y_train.value_counts()))
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print("Class counts after SMOTE: ", dict(pd.Series(y_train_res).value_counts()))

xgb_model = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    eval_metric="logloss", tree_method="hist", random_state=42
)

# SVM (kernel='rbf') is very slow on 200k+ rows, so it's trained on a
# smaller stratified subsample instead -- a standard real-world compromise.
svm_sample_size = 20000
legit_idx = y_train[y_train == 0].sample(n=svm_sample_size // 2, random_state=42).index
fraud_idx = y_train[y_train == 1].index  # take all fraud rows, there are few
sub_idx = legit_idx.append(fraud_idx)
X_sub = X_train_scaled[y_train.index.get_indexer(sub_idx)]
y_sub = y_train.loc[sub_idx].values

smote_sub = SMOTE(random_state=42)
X_sub_res, y_sub_res = smote_sub.fit_resample(X_sub, y_sub)
print("SVM trained on subsample of", len(X_sub), "rows (balanced to", len(X_sub_res), "after SMOTE)")

svm_model = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)



Train size: 212794  Test size: 70932
Fraud rate — train: 0.00167  test: 0.00166

Class counts before SMOTE: {0: np.int64(212439), 1: np.int64(355)}
Class counts after SMOTE:  {0: np.int64(212439), 1: np.int64(212439)}
SVM trained on subsample of 10355 rows (balanced to 20000 after SMOTE)


In [43]:
# ============================================================================
# STEP 4: TRAINING
# ============================================================================
xgb_model.fit(X_train_res, y_train_res)
svm_model.fit(X_sub_res, y_sub_res)

SVC(class_weight='balanced', probability=True, random_state=42)

In [44]:
# ============================================================================
# STEP 5: ROC-AUC EVALUATION
# ============================================================================
y_prob_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]
y_prob_svm = svm_model.predict_proba(X_test_scaled)[:, 1]

auc_xgb = roc_auc_score(y_test, y_prob_xgb)
auc_svm = roc_auc_score(y_test, y_prob_svm)

print("\n===== ROC-AUC =====")
print("XGBoost:", round(auc_xgb, 4))
print("SVM:    ", round(auc_svm, 4))


===== ROC-AUC =====
XGBoost: 0.9766
SVM:     0.9798


In [45]:
# ============================================================================
# STEP 6: FALSE NEGATIVE ANALYSIS + threshold tuning + feature importance
# ============================================================================
threshold = 0.5
y_pred_xgb = (y_prob_xgb >= threshold).astype(int)
cm = confusion_matrix(y_test, y_pred_xgb)
tn, fp, fn, tp = cm.ravel()

print("\n===== XGBoost (threshold =", threshold, ") =====")
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm)
print("False Negatives (fraud MISSED):     ", fn)
print("False Positives (legit txn flagged):", fp)
print("True Positives (fraud caught):      ", tp)
print("True Negatives (legit cleared):     ", tn)
print(classification_report(y_test, y_pred_xgb, target_names=["Legit", "Fraud"], zero_division=0))

print("Business cost discussion:")
print("- Only", round(fraud_rate * 100, 3), "% of transactions are fraud, so even a small")
print("  False Positive RATE means a large false positive COUNT.")
print("- Missing a fraud (FN) usually costs the bank the stolen amount;")
print("  a false alarm (FP) just costs a customer-service call.")
print("- That's why fraud teams often pick a LOWER threshold to catch more")
print("  fraud, accepting more manual reviews.")

# Threshold tuning: sweep a few thresholds to see the FN vs FP trade-off
print("\n--- Threshold tuning ---")
print("Threshold |  FN |   FP |  TP | Recall | Precision")
for t in [0.5, 0.3, 0.2, 0.1, 0.05]:
    y_pred_t = (y_prob_xgb >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    recall_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) else 0
    precision_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) else 0
    print(f"{t:9} | {fn_t:3} | {fp_t:4} | {tp_t:3} | {recall_t:6.3f} | {precision_t:9.3f}")

# Feature importance (top 15)
importances = xgb_model.feature_importances_
order = np.argsort(importances)[::-1][:15]
print("\nTop 15 features driving fraud predictions:")
for i in order:
    print(" ", feature_names[i], "importance=", round(importances[i], 4))

plt.figure(figsize=(8, 6))
plt.barh(np.array(feature_names)[order][::-1], importances[order][::-1])
plt.xlabel("Importance")
plt.title("XGBoost Feature Importance — Fraud Detection (top 15)")
plt.tight_layout()
plt.savefig("fraud_feature_importance_simple.png", dpi=150)
print("\nFeature importance plot saved to fraud_feature_importance_simple.png")

# ROC curve plot for both models
plt.figure(figsize=(7, 6))
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_prob_svm)
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC={auc_xgb:.4f})")
plt.plot(fpr_svm, tpr_svm, label=f"SVM baseline (AUC={auc_svm:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve — Credit Card Fraud Detection")
plt.legend()
plt.tight_layout()
plt.savefig("fraud_roc_curve_simple.png", dpi=150)
print("ROC curve saved to fraud_roc_curve_simple.png")



===== XGBoost (threshold = 0.5 ) =====
Confusion matrix [[TN, FP], [FN, TP]]:
[[70742    72]
 [   23    95]]
False Negatives (fraud MISSED):      23
False Positives (legit txn flagged): 72
True Positives (fraud caught):       95
True Negatives (legit cleared):      70742
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     70814
       Fraud       0.57      0.81      0.67       118

    accuracy                           1.00     70932
   macro avg       0.78      0.90      0.83     70932
weighted avg       1.00      1.00      1.00     70932

Business cost discussion:
- Only 0.167 % of transactions are fraud, so even a small
  False Positive RATE means a large false positive COUNT.
- Missing a fraud (FN) usually costs the bank the stolen amount;
  a false alarm (FP) just costs a customer-service call.
- That's why fraud teams often pick a LOWER threshold to catch more
  fraud, accepting more manual reviews.

--- Threshold tuning ---
Th